# TEMPO — Full EM on Trajectories

**TEMPO** fits a mixture of $M$ regime-specific bases via EM.
Each unit of clustering is a **full spatiotemporal trajectory** $\mathbf{s}^n \in \mathbb{R}^{N_t N_x}$.

EM alternates between:
- **E-step**: recompute soft regime assignments $\gamma_{nm}$ from reconstruction errors
- **M-step**: update mixture weights $\pi_m$ and retrain bases with new $\gamma$

In [ ]:
import os, sys, pathlib, json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from umap import UMAP

sys.path.insert(0, str(pathlib.Path("..").resolve()))
from models.tempo import TEMPOTrainer, TEMPOConfig, pod_factory, fourier_pod_factory
from models.pod import PODConfig
from models.fourier_neural_pod import FourierNeuralPODConfig
from models.tempo_online import build_tempo_online, TEMPOOnlineConfig, _num_modes
from utils.datasets import load_stacked

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

DATA_DIR = os.path.expanduser("~/data/1D/Burgers/Train")


def draw_cov_ellipse(ax, mean2d, cov2d, color, n_std=2.0, **kwargs):
    vals, vecs = np.linalg.eigh(cov2d)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(np.abs(vals))
    ax.add_patch(Ellipse(mean2d, w, h, angle=angle,
                         edgecolor=color, facecolor="none", lw=2, **kwargs))

### Configuration

In [ ]:
# Run
RUN_NAME = "tempo_pod_M3_v1"
RUN_DIR = os.path.join("../../TEMPO_results", RUN_NAME)

# Data
NU_VALUES = [0.001, 0.1, 1.0]
N_SAMPLES = 5000
N_TEST_PER_NU = 1000

# Misc
SEED = 42
N_VIZ = 3
N_UMAP = 5632

os.makedirs(RUN_DIR, exist_ok=True)
print(f"Run dir: {os.path.abspath(RUN_DIR)}")

In [ ]:
REGIME_COLORS = [plt.cm.tab10(i) for i in range(10)]
NU_CMAP = plt.cm.plasma

In [ ]:
entries = [
    (nu, os.path.join(DATA_DIR, f"1D_Burgers_Sols_Nu{nu}.hdf5"))
    for nu in NU_VALUES
]
s_np, kappa_np, x_np, t_np, Nx, Nt = load_stacked(entries, n_samples=N_SAMPLES)
Ny = Nt * Nx

x_grid = torch.tensor(x_np, dtype=torch.float32)
t_grid = torch.tensor(t_np, dtype=torch.float32)
tt, xx = torch.meshgrid(t_grid, x_grid, indexing='ij')
x_flat = torch.stack([xx.flatten(), tt.flatten()], dim=1)  # (Ny, 2)

s = torch.from_numpy(s_np); del s_np
kappa = torch.from_numpy(kappa_np[:, None]); del kappa_np
x = x_flat.to(DEVICE)

print(f"s={s.shape}, x={x.shape}, kappa={kappa.shape}")

In [ ]:
cfg = TEMPOConfig(
    M=3,
    P_global=25,
    sigma2=0.1,
    max_em_iters=30,
    eps_skip=0.01,
    eps_large=0.1,
    eps_conv=0.005,
    basis_config=PODConfig(max_modes=32),
    basis_factory=pod_factory,
)

# cfg = TEMPOConfig(
#     M=3, P_global=20, sigma2=0.1, max_em_iters=10,
#     eps_skip=0.01, eps_large=0.1, eps_conv=0.005,
#     basis_config=FourierNeuralPODConfig(max_modes=5, n_epochs_mean=165, n_epochs_mode=85),
#     basis_factory=fourier_pod_factory,
# )

regime_colors = REGIME_COLORS[:cfg.M]

In [5]:
trainer = TEMPOTrainer(cfg)
trainer.train(s, x, t=None, kappa=kappa)

=== Phase 1: global POD ===
POD | N=15000, Ny=205824 | max_modes=25, tol=1.0
  P=25 modes (100.00% variance, needed 35)
  mode  1: sigma=2.7121e+02  cumvar=95.22%
  mode  2: sigma=2.4171e+01  cumvar=95.98%
  mode  3: sigma=2.3579e+01  cumvar=96.70%
  mode  4: sigma=1.5947e+01  cumvar=97.03%
  mode  5: sigma=1.5662e+01  cumvar=97.34%
  mode  6: sigma=1.5405e+01  cumvar=97.65%
  mode  7: sigma=1.4625e+01  cumvar=97.93%
  mode  8: sigma=1.1294e+01  cumvar=98.09%
  mode  9: sigma=1.1207e+01  cumvar=98.25%
  mode 10: sigma=1.0933e+01  cumvar=98.41%
  mode 11: sigma=1.0402e+01  cumvar=98.55%
  mode 12: sigma=1.0278e+01  cumvar=98.69%
  mode 13: sigma=1.0098e+01  cumvar=98.82%
  mode 14: sigma=1.0063e+01  cumvar=98.95%
  mode 15: sigma=9.8948e+00  cumvar=99.08%
  mode 16: sigma=8.2250e+00  cumvar=99.16%
  mode 17: sigma=8.0585e+00  cumvar=99.25%
  mode 18: sigma=7.5756e+00  cumvar=99.32%
  mode 19: sigma=7.4151e+00  cumvar=99.39%
  mode 20: sigma=6.4805e+00  cumvar=99.45%
  mode 21: sigma=6.4

  done
  regime 3: full rerun (delta=1.112e+00)
POD | N=15000, Ny=205824 | max_modes=32, tol=0.9999
  P=32 modes (99.99% variance, needed 39)
  mode  1: sigma=2.8693e+02  cumvar=99.07%
  mode  2: sigma=1.1270e+01  cumvar=99.22%
  mode  3: sigma=1.0142e+01  cumvar=99.35%
  mode  4: sigma=7.1342e+00  cumvar=99.41%
  mode  5: sigma=7.0662e+00  cumvar=99.47%
  mode  6: sigma=6.2350e+00  cumvar=99.52%
  mode  7: sigma=6.1084e+00  cumvar=99.56%
  mode  8: sigma=5.8265e+00  cumvar=99.60%
  mode  9: sigma=5.7149e+00  cumvar=99.64%
  mode 10: sigma=4.8365e+00  cumvar=99.67%
  mode 11: sigma=4.6167e+00  cumvar=99.70%
  mode 12: sigma=4.5906e+00  cumvar=99.72%
  mode 13: sigma=4.5234e+00  cumvar=99.75%
  mode 14: sigma=4.2766e+00  cumvar=99.77%
  mode 15: sigma=4.2537e+00  cumvar=99.79%
  mode 16: sigma=3.8358e+00  cumvar=99.81%
  mode 17: sigma=3.7225e+00  cumvar=99.82%
  mode 18: sigma=3.2699e+00  cumvar=99.84%
  mode 19: sigma=3.1999e+00  cumvar=99.85%
  mode 20: sigma=3.1789e+00  cumvar=99.86

  done
  regime 3: full rerun (delta=3.624e-01)
POD | N=15000, Ny=205824 | max_modes=32, tol=0.9999
  P=32 modes (99.99% variance, needed 42)
  mode  1: sigma=2.6377e+02  cumvar=96.68%
  mode  2: sigma=2.0007e+01  cumvar=97.24%
  mode  3: sigma=1.9645e+01  cumvar=97.77%
  mode  4: sigma=1.3201e+01  cumvar=98.02%
  mode  5: sigma=1.2992e+01  cumvar=98.25%
  mode  6: sigma=1.1454e+01  cumvar=98.43%
  mode  7: sigma=1.1114e+01  cumvar=98.60%
  mode  8: sigma=9.3617e+00  cumvar=98.73%
  mode  9: sigma=9.2327e+00  cumvar=98.84%
  mode 10: sigma=8.5777e+00  cumvar=98.95%
  mode 11: sigma=8.4527e+00  cumvar=99.05%
  mode 12: sigma=7.6831e+00  cumvar=99.13%
  mode 13: sigma=7.5598e+00  cumvar=99.21%
  mode 14: sigma=7.4122e+00  cumvar=99.28%
  mode 15: sigma=7.2041e+00  cumvar=99.36%
  mode 16: sigma=5.7802e+00  cumvar=99.40%
  mode 17: sigma=5.4960e+00  cumvar=99.44%
  mode 18: sigma=5.3831e+00  cumvar=99.48%
  mode 19: sigma=5.2605e+00  cumvar=99.52%
  mode 20: sigma=4.9629e+00  cumvar=99.56

  done
  regime 3: full rerun (delta=3.222e-01)
POD | N=15000, Ny=205824 | max_modes=32, tol=0.9999
  P=32 modes (99.99% variance, needed 42)
  mode  1: sigma=2.4754e+02  cumvar=95.06%
  mode  2: sigma=2.2929e+01  cumvar=95.87%
  mode  3: sigma=2.2473e+01  cumvar=96.66%
  mode  4: sigma=1.5657e+01  cumvar=97.04%
  mode  5: sigma=1.5545e+01  cumvar=97.41%
  mode  6: sigma=1.2899e+01  cumvar=97.67%
  mode  7: sigma=1.2592e+01  cumvar=97.91%
  mode  8: sigma=1.1049e+01  cumvar=98.10%
  mode  9: sigma=1.0939e+01  cumvar=98.29%
  mode 10: sigma=1.0174e+01  cumvar=98.45%
  mode 11: sigma=9.9618e+00  cumvar=98.60%
  mode 12: sigma=9.1081e+00  cumvar=98.73%
  mode 13: sigma=8.8897e+00  cumvar=98.86%
  mode 14: sigma=8.2395e+00  cumvar=98.96%
  mode 15: sigma=8.2254e+00  cumvar=99.07%
  mode 16: sigma=6.3970e+00  cumvar=99.13%
  mode 17: sigma=6.3317e+00  cumvar=99.19%
  mode 18: sigma=6.1289e+00  cumvar=99.25%
  mode 19: sigma=5.9238e+00  cumvar=99.30%
  mode 20: sigma=5.8210e+00  cumvar=99.36

  done

─────────────────────────────────────────────────────────────────
E-step 8...
EM   8 | LL=-1.9585e+04 | BIC=4.9286e+04 | H=0.898
       | delta=[1.018e-02 5.443e-03 1.969e+00] | pi=[0.447 0.346 0.206]
M2-step: updating bases...
  regime 1: incremental (delta=1.018e-02)
POD | N=15000, Ny=205824 | max_modes=32, tol=0.9999
  P=32 modes (99.99% variance, needed 42)
  mode  1: sigma=2.6417e+02  cumvar=94.63%
  mode  2: sigma=2.7176e+01  cumvar=95.63%
  mode  3: sigma=2.4717e+01  cumvar=96.46%
  mode  4: sigma=1.7100e+01  cumvar=96.86%
  mode  5: sigma=1.6571e+01  cumvar=97.23%
  mode  6: sigma=1.4940e+01  cumvar=97.53%
  mode  7: sigma=1.4495e+01  cumvar=97.82%
  mode  8: sigma=1.1712e+01  cumvar=98.00%
  mode  9: sigma=1.1267e+01  cumvar=98.17%
  mode 10: sigma=1.0595e+01  cumvar=98.33%
  mode 11: sigma=1.0397e+01  cumvar=98.47%
  mode 12: sigma=1.0281e+01  cumvar=98.62%
  mode 13: sigma=1.0152e+01  cumvar=98.76%
  mode 14: sigma=1.0028e+01  cumvar=98.89%
  mode 15: sigma=9.6359e+0

  done
  regime 2: skip (delta=3.405e-03)
  regime 3: full rerun (delta=3.996e-01)
POD | N=15000, Ny=205824 | max_modes=32, tol=0.9999
  P=32 modes (99.99% variance, needed 42)
  mode  1: sigma=2.3691e+02  cumvar=93.51%
  mode  2: sigma=2.6336e+01  cumvar=94.66%
  mode  3: sigma=2.5441e+01  cumvar=95.74%
  mode  4: sigma=1.7567e+01  cumvar=96.26%
  mode  5: sigma=1.7357e+01  cumvar=96.76%
  mode  6: sigma=1.3977e+01  cumvar=97.08%
  mode  7: sigma=1.3306e+01  cumvar=97.38%
  mode  8: sigma=1.2421e+01  cumvar=97.64%
  mode  9: sigma=1.2221e+01  cumvar=97.88%
  mode 10: sigma=1.1039e+01  cumvar=98.09%
  mode 11: sigma=1.0761e+01  cumvar=98.28%
  mode 12: sigma=1.0202e+01  cumvar=98.45%
  mode 13: sigma=9.8472e+00  cumvar=98.61%
  mode 14: sigma=8.4797e+00  cumvar=98.73%
  mode 15: sigma=8.4108e+00  cumvar=98.85%
  mode 16: sigma=7.0557e+00  cumvar=98.94%
  mode 17: sigma=6.8321e+00  cumvar=99.01%
  mode 18: sigma=6.7615e+00  cumvar=99.09%
  mode 19: sigma=6.7257e+00  cumvar=99.16%
  mode

  done

─────────────────────────────────────────────────────────────────
E-step 14...
EM  14 | LL=-1.9548e+04 | BIC=4.9212e+04 | H=0.909
       | delta=[4.815e-03 3.519e-03 6.183e-02] | pi=[0.429 0.333 0.238]
M2-step: updating bases...
  regime 1: skip (delta=4.815e-03)
  regime 2: skip (delta=3.519e-03)
  regime 3: incremental (delta=6.183e-02)
POD | N=15000, Ny=205824 | max_modes=32, tol=0.9999
  P=32 modes (99.99% variance, needed 42)
  mode  1: sigma=2.3715e+02  cumvar=93.54%
  mode  2: sigma=2.6367e+01  cumvar=94.70%
  mode  3: sigma=2.5390e+01  cumvar=95.77%
  mode  4: sigma=1.7522e+01  cumvar=96.28%
  mode  5: sigma=1.7320e+01  cumvar=96.78%
  mode  6: sigma=1.3981e+01  cumvar=97.10%
  mode  7: sigma=1.3266e+01  cumvar=97.40%
  mode  8: sigma=1.2390e+01  cumvar=97.65%
  mode  9: sigma=1.2206e+01  cumvar=97.90%
  mode 10: sigma=1.1025e+01  cumvar=98.10%
  mode 11: sigma=1.0755e+01  cumvar=98.30%
  mode 12: sigma=1.0173e+01  cumvar=98.47%
  mode 13: sigma=9.8255e+00  cumvar=98.63

### Results — EM regime structure

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(s), min(N_UMAP, len(s)), replace=False)

kappa_viz = kappa[idx, 0].cpu().numpy()
alpha_viz = trainer.alpha[idx].cpu().numpy()
hard_labels = trainer.gamma[idx].argmax(dim=1).cpu().numpy()
mu_np = trainer.mu.cpu().numpy()  # (M, P_global)

UMAP of trajectory coefficients $\alpha^n \in \mathbb{R}^P$:

In [ ]:
reducer = UMAP(n_neighbors=30, min_dist=0.0)
embedding = reducer.fit_transform(alpha_viz)
mu_umap = reducer.transform(mu_np)

In [ ]:
nu_unique = np.unique(kappa_viz)
nu_to_col = {nu: NU_CMAP(i / max(len(nu_unique) - 1, 1))
             for i, nu in enumerate(nu_unique)}
point_colors_reg = [regime_colors[m] for m in hard_labels]
point_colors_nu  = [nu_to_col[float(nu)] for nu in kappa_viz]

centroid_kw = dict(s=120, marker="D", zorder=10, edgecolors="white", linewidths=0.8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(embedding[:, 0], embedding[:, 1],
                c=point_colors_reg, s=3, alpha=0.3, rasterized=True, linewidths=0)
for m in range(cfg.M):
    axes[0].scatter(mu_umap[m, 0], mu_umap[m, 1],
                    color=regime_colors[m], **centroid_kw)
axes[0].set_title("UMAP — EM regime assignment", fontweight="bold")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=regime_colors[m],
                      markersize=7, label=f"Regime {m+1}") for m in range(cfg.M)]
axes[0].legend(handles=handles, fontsize=9, framealpha=0.7)

axes[1].scatter(embedding[:, 0], embedding[:, 1],
                c=point_colors_nu, s=3, alpha=0.3, rasterized=True, linewidths=0)
for m in range(cfg.M):
    axes[1].scatter(mu_umap[m, 0], mu_umap[m, 1],
                    color=regime_colors[m], **centroid_kw)
axes[1].set_title(r"UMAP — colored by $\nu$", fontweight="bold")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=nu_to_col[nu],
                      markersize=7, label=rf"$\nu={nu:.3f}$") for nu in nu_unique]
axes[1].legend(handles=handles, fontsize=9, framealpha=0.7)

axes[2].scatter(alpha_viz[:, 0], alpha_viz[:, 1],
                c=point_colors_nu, s=3, alpha=0.3, rasterized=True, linewidths=0)
for m in range(cfg.M):
    cov2d = trainer.Sigma[m, :2, :2].cpu().numpy()
    draw_cov_ellipse(axes[2], mu_np[m, :2], cov2d, color=regime_colors[m], linestyle="--")
    axes[2].scatter(*mu_np[m, :2], color=regime_colors[m], **centroid_kw,
                    label=f"Regime {m+1}")
axes[2].set_title(r"POD space: $\alpha_1$ vs $\alpha_2$", fontweight="bold")
axes[2].set_xlabel(r"$\alpha_1$"); axes[2].set_ylabel(r"$\alpha_2$")
axes[2].legend(fontsize=9, framealpha=0.7)

for ax in axes[:2]:
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
for ax in axes:
    ax.grid(True, ls="--", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Burgers trajectories — TEMPO EM regime structure",
             fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "TEMPO_phase1.png"), dpi=150, bbox_inches="tight")
plt.show()

## Phase 2 — Online Gated Operator

Bases $\{\Phi_m^*\}$ and responsibilities $\{\gamma_{im}^*\}$ are frozen.
We train a **GatingNet** $\mathcal{W}(u_0, \kappa) \to \Delta^{M-1}$ and $M$ **BranchNets** $\mathcal{B}_m(u_0, \kappa) \to \mathbb{R}^{P_m}$.

Loss: $\mathcal{L} = \mathcal{L}_{\mathrm{data}} + \lambda_{\mathrm{KL}}\,\mathcal{L}_{\mathrm{KL}} - \lambda_H\,\mathcal{L}_{\mathrm{ent}}$

In [ ]:
N_per_nu = N_SAMPLES

train_idx = torch.cat([torch.arange(i*N_per_nu, (i+1)*N_per_nu - N_TEST_PER_NU)
                       for i in range(len(NU_VALUES))])
test_idx = torch.cat([torch.arange((i+1)*N_per_nu - N_TEST_PER_NU, (i+1)*N_per_nu)
                      for i in range(len(NU_VALUES))])

# Initial condition: first spatial slice (t = 0)
u0 = s[:, :Nx]  # (N_total, Nx)

s_train = s[train_idx];         s_test = s[test_idx]
u0_train = u0[train_idx];       u0_test = u0[test_idx]
kappa_train = kappa[train_idx]; kappa_test = kappa[test_idx]
gamma_train = trainer.gamma[train_idx]

print(f'train: {s_train.shape}, test: {s_test.shape}')

In [10]:
cfg_online = TEMPOOnlineConfig(
    lr=3e-4,
    n_epochs=170,
    batch_size=16,
    hidden_dim=128,
    n_layers=4,
    sensor_stride=2,
    lambda_kl=0.1,
    lambda_ent=0.1,
    log_every=20,
)

model_online, online_trainer = build_tempo_online(
    trainers=trainer.trainers,
    d_kappa=kappa.shape[1],
    Nx=Nx,
    cfg=cfg_online,
)
model_online = model_online.to(DEVICE)

for m, t in enumerate(trainer.trainers):
    print(f'Regime {m+1}: P={_num_modes(t)} modes')

Regime 1: P=32 modes
Regime 2: P=32 modes
Regime 3: P=32 modes


In [11]:
history = online_trainer.train(
    s=s_train,
    u0=u0_train,
    kappa=kappa_train,
    x_flat=x,
    gamma_star=gamma_train,
    trainers=trainer.trainers,
)

  epoch    0 | loss=1.7770e-01 | data=2.3572e-01  kl=2.3629e-01  ent=8.1648e-01
  epoch   40 | loss=-8.1588e-02 | data=1.4282e-02  kl=4.4112e-02  ent=1.0028e+00
  epoch   60 | loss=-8.4806e-02 | data=1.1289e-02  kl=4.2834e-02  ent=1.0038e+00
  epoch   80 | loss=-8.6173e-02 | data=9.7454e-03  kl=4.3929e-02  ent=1.0031e+00
  epoch  100 | loss=-8.7713e-02 | data=8.2772e-03  kl=4.2953e-02  ent=1.0029e+00
  epoch  120 | loss=-8.8717e-02 | data=7.7117e-03  kl=3.8767e-02  ent=1.0031e+00
  epoch  140 | loss=-8.7964e-02 | data=7.9848e-03  kl=4.3564e-02  ent=1.0030e+00
  epoch  160 | loss=-8.9514e-02 | data=6.9751e-03  kl=3.8199e-02  ent=1.0031e+00
  done: final loss=-9.0015e-02


### Training dynamics

In [ ]:
epochs = range(len(history['total']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

c0, c1 = plt.cm.tab10(0), plt.cm.tab10(1)

axes[0].plot(epochs, history['total'], color=c0, label='total', lw=2)
axes[0].plot(epochs, history['data'],  color=c1, label=r'$\mathcal{L}_{\mathrm{data}}$', lw=1.5, ls='--')
axes[0].set_title('Total loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(framealpha=0.7); axes[0].set_yscale('log')

axes[1].plot(epochs, history['kl'],  color=plt.cm.tab10(2), label=r'$\mathcal{L}_{\mathrm{KL}}$',  lw=1.5)
axes[1].plot(epochs, history['ent'], color=plt.cm.tab10(3), label=r'$\mathcal{L}_{\mathrm{ent}}$', lw=1.5, ls='--')
axes[1].set_title('Regularisation terms', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend(framealpha=0.7)

for ax in axes:
    ax.grid(True, ls='--', alpha=0.25)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "training_dynamics.png"), dpi=150, bbox_inches="tight")
plt.show()

### Gating & reconstruction quality on test set

In [ ]:
s_pred, w_pred = online_trainer.predict(
    u0_new=u0_test, kappa_new=kappa_test,
    x_flat=x, trainers=trainer.trainers,
)

s_pred_np = s_pred.cpu().numpy()
s_test_np = s_test.numpy()
w_np = w_pred.cpu().numpy()  # (N_test, M)
kappa_t_np = kappa_test[:, 0].numpy()
nu_unique_t = np.unique(kappa_t_np)

rel_l2 = (np.linalg.norm(s_pred_np - s_test_np, axis=1)
          / np.linalg.norm(s_test_np, axis=1))
print('Mean rel L2 error:')
for nu in nu_unique_t:
    mask = kappa_t_np == nu
    print(f'  nu={nu:.3f}: {rel_l2[mask].mean():.4f} +/- {rel_l2[mask].std():.4f}')

In [ ]:
nu_unique_t = np.unique(kappa_t_np)
nu_to_col_t = {nu: NU_CMAP(i / max(len(nu_unique_t) - 1, 1))
               for i, nu in enumerate(nu_unique_t)}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x_pos = np.arange(len(nu_unique_t))
width = 0.8 / cfg.M
for m in range(cfg.M):
    w_m = [w_np[kappa_t_np == nu, m].mean() for nu in nu_unique_t]
    axes[0].bar(x_pos + m * width, w_m, width,
                color=regime_colors[m], label=f'Regime {m+1}', alpha=0.85, linewidth=0)
axes[0].set_xticks(x_pos + width * (cfg.M - 1) / 2)
axes[0].set_xticklabels([rf'$\nu={nu:.3f}$' for nu in nu_unique_t])
axes[0].set_ylabel('Mean gating weight')
axes[0].set_title('Gating weights per viscosity', fontweight='bold')
axes[0].legend(fontsize=9, framealpha=0.7)
axes[0].set_ylim(0, 1)

bp_data = [rel_l2[kappa_t_np == nu] for nu in nu_unique_t]
bp = axes[1].boxplot(bp_data, patch_artist=True, widths=0.45,
                     medianprops=dict(color="black", lw=1.5),
                     whiskerprops=dict(lw=1.2),
                     capprops=dict(lw=1.2),
                     flierprops=dict(marker="o", markersize=3, alpha=0.4, linestyle="none"))
for patch, nu in zip(bp['boxes'], nu_unique_t):
    patch.set_facecolor(nu_to_col_t[nu])
    patch.set_alpha(0.8)
    patch.set_linewidth(0)
axes[1].set_xticklabels([rf'$\nu={nu:.3f}$' for nu in nu_unique_t])
axes[1].set_ylabel('Relative $L^2$ error')
axes[1].set_title('Reconstruction error by viscosity', fontweight='bold')
axes[1].set_ylim(0, 1)

for ax in axes:
    ax.grid(True, ls='--', alpha=0.25, axis='y')
    ax.spines[['top', 'right']].set_visible(False)
plt.suptitle('TEMPO Phase 2 — test set results', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "TEMPO_phase2.png"), dpi=150, bbox_inches="tight")
plt.show()

### Reconstruction examples

In [ ]:
fig, axes = plt.subplots(len(nu_unique_t), 3,
                         figsize=(13, 3.5 * len(nu_unique_t)))

for row, nu in enumerate(nu_unique_t):
    idx_nu = np.where(kappa_t_np == nu)[0][0]
    s_true  = s_test_np[idx_nu].reshape(Nt, Nx)
    s_hat_r = s_pred_np[idx_nu].reshape(Nt, Nx)
    err = np.abs(s_true - s_hat_r)
    vmin, vmax = s_true.min(), s_true.max()

    kw = dict(aspect='auto', origin='lower',
              extent=[x_np.min(), x_np.max(), t_np.min(), t_np.max()])
    axes[row, 0].imshow(s_true,   vmin=vmin, vmax=vmax, cmap='RdBu_r', **kw)
    axes[row, 1].imshow(s_hat_r,  vmin=vmin, vmax=vmax, cmap='RdBu_r', **kw)
    im = axes[row, 2].imshow(err, cmap='Oranges', **kw)
    plt.colorbar(im, ax=axes[row, 2], fraction=0.046)

    axes[row, 0].set_title(rf'$\nu={nu:.3f}$ — ground truth')
    axes[row, 1].set_title(rf'$\nu={nu:.3f}$ — TEMPO')
    axes[row, 2].set_title(rf'$\nu={nu:.3f}$ — |error|')
    for ax in axes[row]:
        ax.set_xlabel('x'); ax.set_ylabel('t')
        ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Space-time reconstructions — TEMPO(POD)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "reconstruct.png"), dpi=150, bbox_inches="tight")
plt.show()

### Metrics

In [ ]:
metrics = {
    "run_name":       RUN_NAME,
    "M":              int(cfg.M),
    "n_train":        int(len(s_train)),
    "n_test":         int(len(s_test)),
    "overall_mean":   float(rel_l2.mean()),
    "overall_median": float(np.median(rel_l2)),
}
for nu in nu_unique_t:
    mask = kappa_t_np == nu
    metrics[f"nu{nu:.3f}_mean"]   = float(rel_l2[mask].mean())
    metrics[f"nu{nu:.3f}_median"] = float(np.median(rel_l2[mask]))
    metrics[f"nu{nu:.3f}_std"]    = float(rel_l2[mask].std())

# Training logs
metrics["phase1_log"] = trainer.history_phase1
metrics["phase2_log"] = history

print(metrics)

### Checkpoint

In [ ]:
torch.save({
    'model_online': model_online.state_dict(),
    'cfg': cfg,
    'cfg_online': cfg_online,
    'metrics': metrics,
    'run_name': RUN_NAME,
}, os.path.join(RUN_DIR, "model_online.pt"))

torch.save(trainer, os.path.join(RUN_DIR, "trainer.pt"))

with open(os.path.join(RUN_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved to {os.path.abspath(RUN_DIR)}")